[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/notebooks/learn/02_address_and_read.ipynb)

In [1]:
# Colab setup. Does nothing when this notebook runs from a TorchLens checkout
# (learn_models.py is beside it). On Colab there is no checkout, so this installs
# TorchLens from the main branch and fetches learn_models.py next to the notebook.
import pathlib
import shutil
import subprocess
import sys
import urllib.request

if not pathlib.Path("learn_models.py").exists():
    REPO = "johnmarktaylor91/torchlens"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torchlens[viz] @ git+https://github.com/{REPO}@main",
            "transformers",
            "ipywidgets",
        ],
        check=True,
    )
    if shutil.which("dot") is None:  # drawings need the Graphviz binary
        subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{REPO}/main/notebooks/learn/learn_models.py",
        "learn_models.py",
    )

# Address and read

**Level 1 of 5. Starting point: you want a number out.**

Every operation the forward ran has a label, and `trace[label]` is its record: the activation,
its shape and dtype, the function that made it, the module it ran in, its neighbours in the
dataflow and the source line that wrote it. This notebook is the label grammar, the keys that
reach a record, what a record carries, and what happens when a model returns a dict instead of a
tensor.

> **You'll want to know:** [`tl.trace` and the summary](01_capture_and_look.ipynb) (one call
> records a forward pass), [the Trace is read like a dict](00_quickstart.ipynb) (a label in
> square brackets), and that a PyTorch model may return a dict or a tuple, not only a tensor.

In [2]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [3]:
from learn_models import tiny_dict_model, tiny_mlp, tiny_rnn  # tiny seeded CPU models

model, x = tiny_mlp()  # two linear layers with a ReLU between them, and a (4, 8) input batch

The ReLU's record, printed, and the activation itself.

In [4]:
trace = tl.trace(model, x)
relu = trace["relu_1_2"]
print(relu)
relu.out[0, :6].detach()  # the activation: first batch item, first six units

Layer relu_1: float32 [4, 16] on cpu, min 0.000 max 1.204 mean 0.2228 sd 0.3267
  position  operation 2 of 3
  output    64 values, 256 B
  values    min 0.000  max 1.204  mean 0.2228  sd 0.3267
  graph     linear_1 -> relu_1 -> linear_2 (reaches 1 of 1 model output)
  function  relu, gradient ReluBackward0, 92.0 us
  source    learn_models.py:32 in TinyMLP.forward
  more      .ops  .parents  .children  .to_pandas()


tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0436, 0.0000])

## 1. The label grammar

A layer label has three parts. `relu_1_2` is a `relu`, the first relu in the model, the second
operation overall. Op labels add a pass suffix, `relu_1_2:1`, which matters for layers that run
more than once (section 4). The trace keeps both lists, and three accessors group the records:
`trace.layers` by place in the code, `trace.ops` by execution, `trace.modules` by module.
`count_by` tallies a collection by a field; the input and output records have kind `none`, they
are bookkeeping, not computation.

In [5]:
print("layers:", trace.layer_labels)
print("ops   :", trace.op_labels)
trace.layers.count_by("kind")

layers: ['input_1', 'linear_1_1', 'relu_1_2', 'linear_2_3', 'output_1']
ops   : ['input_1:1', 'linear_1_1:1', 'relu_1_2:1', 'linear_2_3:1', 'output_1:1']


{'none': 2, 'linear': 2, 'relu': 1}

## 2. Four keys that reach the same record

Square brackets take a full label, a position, any unique substring, or a pass-qualified op
label. A module name is a key too, and it gives the module itself: its class, how often it was
called, and the layers that ran inside it (the same record as `trace.modules[name]`). What you
usually want from a module is what came out of it; `trace.ops[name]` gives that output operation.

In [6]:
keys = [trace["relu_1_2"], trace[2], trace["relu"], trace.ops["relu_1_2:1"]]
print("one record, four keys:", [op.layer_label for op in keys])
first = trace.ops["in_proj"]  # .ops turns a module name into its output op
print("in_proj's output op:", first.layer_label, "| ran in module", first.module)
module = trace["in_proj"]  # the bare module name gives the Module record itself
print(
    "the module:", module.class_name, "| calls:", module.num_calls, "| layers:", module.layer_labels
)

one record, four keys: ['relu_1_2', 'relu_1_2', 'relu_1_2', 'relu_1_2']
in_proj's output op: linear_1_1 | ran in module in_proj:1
the module: Linear | calls: 1 | layers: ['linear_1_1']


## 3. A teaching error

A key that matches two records raises an error rather than guess, and names both. A key that matches
nothing says so and suggests the closest labels. `trace[tl.label_contains(text)]` is the search you
reach for when you half-remember a name.

In [7]:
try:
    trace["linear"]
except tl.errors.AmbiguousOpLookupError as error:
    print(error)
try:
    trace["relu_9"]
except tl.errors.InvalidArgumentError as error:
    print(error)
trace[tl.label_contains("lin")].labels

Ambiguous lookup: 'linear' matches 2 layers.
  candidates 'linear_1', 'linear_2'
  remedy     use a pass-qualified label, full address, or explicit call index
Layer 'relu_9' not found.
  candidates 'relu_1', 'input_1', 'output_1', 'linear_2', 'linear_1'
  remedy     pass an in-range integer, a layer label, or a module address


('linear_1_1:1', 'linear_2_3:1')

## 4. Passes

A layer is one place in the code. When that place runs again, in a loop, the layer has several
passes and each pass is its own op: `linear_1_1:2` is the second pass alone, and the layer with
all three is `trace.layers["linear_1_1"]`. In square brackets on the trace, the bare label returns
that whole `Layer` rather than guessing a pass; asking it for one pass's value raises and names
the spellings that would work. Each layer's `num_passes` says how often it ran.

In [8]:
rnn, seq = tiny_rnn()  # one linear cell and a tanh, applied three times in a Python loop
loop = tl.trace(rnn, seq)
cell = loop.layers["linear_1_1"]  # the aggregate Layer: all three passes
print("bare label gives:", type(loop["linear_1_1"]).__name__)  # the whole layer, not a guessed pass
print(
    type(cell).__name__,
    "with",
    cell.num_passes,
    "passes |",
    {layer.layer_label: layer.num_passes for layer in loop.layers},
)
print("pass 2:", loop["linear_1_1:2"].out[0, :4].detach())
print("pass 3:", loop["linear_1_1:3"].out[0, :4].detach())

bare label gives: Layer
Layer with 3 passes | {'input_1': 1, 'linear_1_1': 3, 'tanh_1_2': 3, 'output_1': 1}
pass 2: tensor([0.3678, 0.5069, 0.0547, 0.0121])
pass 3: tensor([-0.1741,  0.4708, -0.0405,  0.0452])


## 5. What a record carries

Beyond the activation, a record carries its dtype, its cost (forward FLOPs and the bytes the
activation takes), and its neighbours: `parents` and `children` are labels, `get_parents()` gives
the records. Captured with `CaptureOptions(save_source_context=True)`, the trace also knows where the
model's code lives, and each record knows the call site that produced it.

In [9]:
print(
    "dtype",
    relu.dtype,
    "| flops",
    relu.cost.flops_forward,
    "| activation bytes",
    relu.cost.activation_memory,
)
print("first parent record:", relu.get_parents()[0].layer_label)
sourced = tl.trace(model, x, capture=tl.options.CaptureOptions(save_source_context=True))
print(
    "model source:",
    Path(sourced.root_module.class_source_file).name,
    "| forward at line",
    sourced.root_module.forward_source_line,
)
sourced["relu_1_2"].source.frames[-1]  # innermost frame: the model line that produced this op

dtype torch.float32 | flops 64.0 FLOPs | activation bytes 256 B
first parent record: linear_1_1
model source: learn_models.py | forward at line 31


FuncCallLocation: learn_models.py:32 in forward

The same option kept the source text itself with the trace: `call_line` is the line that ran,
snapshotted at capture time, so it still reads correctly if the file changes later. `capture=` takes a
`CaptureOptions`; this is the first of several options you will meet there.

In [10]:
sourced["relu_1_2"].source.frames[-1].call_line  # the source line itself, kept with the trace

'hidden = torch.relu(self.in_proj(x))'

## 6. Containers: a model that returns a dict

Many models return a dict, a tuple or a Hugging Face `ModelOutput`. The trace keeps the
structure. `trace.reconstruct_output()` rebuilds the very object the model returned, with the saved
activations as its leaves, and `trace.output_layers` names the operations that fed it. The
output record's `container` is a view of that structure: index it by key to reach the leaf
record and its activation, or `reconstruct()` the whole object from the saved leaves.

In [11]:
dict_model, dx = tiny_dict_model()  # returns {"logits": ..., "hidden": ...}
dt = tl.trace(dict_model, dx)
print(
    "trace.reconstruct_output() rebuilds the dict:",
    {k: tuple(v.shape) for k, v in dt.reconstruct_output().items()},
)
print("output layers:", dt.output_layers)
view = dt[dt.output_layers[0]].container
print(
    "container:", view.kind, "with", len(view), "entries | 'logits' is", view["logits"].layer_label
)
print("reconstructed:", {k: tuple(v.shape) for k, v in view.reconstruct().items()})
view["logits"].out[0].detach()  # the leaf record's activation: first item's three logits

trace.reconstruct_output() rebuilds the dict: {'logits': (4, 3), 'hidden': (4, 16)}
output layers: ['output_1', 'output_2']
container: dict with 2 entries | 'logits' is output_1
reconstructed: {'logits': (4, 3), 'hidden': (4, 16)}


tensor([-0.6131,  0.0733, -0.1550])

## 7. Where this leaves you

Every operation has a label you can read, four ways to type it, and a record that carries the
activation, its cost, its neighbours and its source line; a model that returns a dict keeps the
dict. The next levels add keywords to `tl.trace` that choose what to keep and what to change.

**One thing about networks:** the ReLU's `parents` and `children` are the dataflow, not the
module tree. The model's code is a stack of modules, but what it computes is a graph of tensors;
TorchLens addresses the graph first and the stack second, and a module's output is an
operation in that graph, which `trace.ops[name]` reaches by the module's name.

**Go deeper:** next level, [04, Keep what you need](04_keep_what_you_need.ipynb), or
[03, See it](03_see_it.ipynb) to draw the graph first. Recipe cards:
[`pluck.ipynb`](../../examples/quick/pluck.ipynb),
[`extract_activations.ipynb`](../../examples/quick/extract_activations.ipynb). Reference: the
glossary entries [Op, Layer and Module](../../docs/reference/glossary.md#graph-records) and
[the grouping guide](../../docs/guides/grouping.md) for the exact pass rule. A return type
TorchLens does not know yet is taught with `tl.types.register_container(MyType, flatten,
unflatten)`; [08, Semantic shortcuts](08_semantic_shortcuts.ipynb) shows one.